# PHIL-TEXT — Faz 3.3: Veri Temizleme

**Amaç:** Ham metinlerden Gutenberg artefaktlarını, duplikatları ve gürültüyü temizleyip kalite-kontrollü bir corpus oluşturmak.

### Adımlar
1. Ham corpus yükle & önceki halini incele
2. Gutenberg artefakt analizi
3. Temizleme pipeline'ı uygula
4. Öncesi / Sonrası karşılaştırma
5. Duplikat analizi
6. Outlier analizi
7. Kalite skoru dağılımı
8. Temiz corpus kaydet

## 1. Kurulum & Yükleme

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data.load_data import load_corpus
from src.data.clean_data import (
    clean_text_pipeline, clean_corpus, save_clean_corpus,
    remove_gutenberg_artifacts, normalize_whitespace,
)

plt.rcParams['figure.dpi'] = 130
sns.set_theme(style='whitegrid', palette='muted')

PHIL_COLORS = {
    'platon': '#4C72B0', 'aristoteles': '#DD8452', 'marcus_aurelius': '#55A868',
    'descartes': '#C44E52', 'spinoza': '#8172B2', 'locke': '#937860',
    'hume': '#DA8BC3', 'kant': '#8C8C8C', 'schopenhauer': '#CCB974', 'nietzsche': '#64B5CD',
}

df_raw = load_corpus('../data/raw')
print(f'Ham corpus: {df_raw.shape[0]} eser, toplam {df_raw["word_count"].sum():,} kelime')

## 2. Gutenberg Artefakt Analizi

In [ ]:
# Her metnin ilk 800 karakterine bak — header kalıpları
print('=== Ham metin başlık örnekleri ===')
for _, row in df_raw.head(4).iterrows():
    print(f'\n--- {row["philosopher"]} / {row["work"]} ---')
    print(repr(row['text'][:400]))
    print()

In [ ]:
# Gutenberg keyword taraması
gutenberg_keywords = ['Project Gutenberg', 'GUTENBERG', 'produced by', 'This eBook']
print('=== Gutenberg Artefakt Taraması ===')
for kw in gutenberg_keywords:
    count = df_raw['text'].str.contains(kw, case=False, regex=False).sum()
    print(f'  "{kw}" içeren eser sayısı: {count}')

In [ ]:
# Bir eserde artefakt öncesi/sonrası göster
sample = df_raw.iloc[0]
cleaned_sample = remove_gutenberg_artifacts(sample['text'])
print(f'Eser: {sample["philosopher"]} / {sample["work"]}')
print(f'Ham uzunluk   : {len(sample["text"]):,} karakter')
print(f'Temiz uzunluk : {len(cleaned_sample):,} karakter')
print(f'Kazanım       : {len(sample["text"]) - len(cleaned_sample):+,} karakter silindi')
print()
print('--- Temiz metnin ilk 300 karakteri ---')
print(repr(cleaned_sample[:300]))

## 3. Temizleme Pipeline Uygula

In [ ]:
df_clean = clean_corpus(df_raw, text_col='text', remove_dups=True, min_words=1000)
print(f'Temiz corpus: {df_clean.shape[0]} eser, toplam {df_clean["word_count"].sum():,} kelime')
print(f'Sütunlar: {list(df_clean.columns)}')
df_clean[['philosopher','work','word_count','quality_score','is_exact_dup','word_count_outlier']].head(8)

## 4. Öncesi / Sonrası Karşılaştırma

In [ ]:
comparison = df_raw[['philosopher','work','word_count']].merge(
    df_clean[['philosopher','work','word_count']].rename(columns={'word_count':'word_count_clean'}),
    on=['philosopher','work']
)
comparison['diff'] = comparison['word_count_clean'] - comparison['word_count']
comparison['diff_pct'] = (comparison['diff'] / comparison['word_count'] * 100).round(2)

print('=== Kelime Sayısı Değişimi (ham vs temiz) ===')
print(comparison[['philosopher','work','word_count','word_count_clean','diff','diff_pct']]
      .sort_values('diff_pct').to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kelime sayısı ham vs temiz
ax = axes[0]
x = range(len(comparison))
ax.bar([i - 0.2 for i in x], comparison['word_count'], width=0.4, label='Ham', color='steelblue', alpha=0.7)
ax.bar([i + 0.2 for i in x], comparison['word_count_clean'], width=0.4, label='Temiz', color='tomato', alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(comparison['work'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Kelime Sayısı')
ax.set_title('Ham vs Temiz Kelime Sayısı')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v/1000)}K'))
ax.legend()

# Değişim yüzdesi
ax = axes[1]
colors = ['green' if v >= 0 else 'red' for v in comparison['diff_pct']]
ax.bar(range(len(comparison)), comparison['diff_pct'], color=colors, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(comparison)))
ax.set_xticklabels(comparison['work'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Değişim (%)')
ax.set_title('Kelime Sayısı Değişim Yüzdesi (Temizlik Etkisi)')

plt.suptitle('Veri Temizleme Etkisi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/cleaning_before_after.png', bbox_inches='tight')
plt.show()

## 5. Duplikat Analizi

In [ ]:
print('=== Duplikat Özeti ===')
print(f'Tam duplikat  : {df_clean["is_exact_dup"].sum()}')
print(f'Yakın duplikat: {df_clean["is_near_dup"].sum()}')
print(f'Temiz corpus  : {(~df_clean["is_exact_dup"]).sum()} eser')

if df_clean['is_exact_dup'].any():
    print('\nDuplikat olan eserler:')
    print(df_clean[df_clean['is_exact_dup']][['philosopher','work','word_count']])
else:
    print('\nHicbir tam duplikat bulunamadi. Corpus temiz.')

## 6. Outlier Analizi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot — outlier işaretli
ax = axes[0]
outlier_mask = df_clean['word_count_outlier']
ax.scatter(df_clean.index[~outlier_mask], df_clean.loc[~outlier_mask, 'word_count'],
           color='steelblue', s=80, zorder=3, label='Normal')
ax.scatter(df_clean.index[outlier_mask], df_clean.loc[outlier_mask, 'word_count'],
           color='red', s=120, marker='*', zorder=4, label='Outlier')
for idx, row in df_clean[outlier_mask].iterrows():
    ax.annotate(f"{row['philosopher']}\n{row['work']}", (idx, row['word_count']),
                textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_ylabel('Kelime Sayısı')
ax.set_title('Outlier Tespiti (IQR Yöntemi)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v/1000)}K'))
ax.legend()

# Kelime sayısı dağılımı + IQR sınırları
ax = axes[1]
q1, q3 = df_clean['word_count'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
ax.hist(df_clean['word_count'], bins=12, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(lower, color='red', linestyle='--', label=f'Alt sınır: {lower:.0f}')
ax.axvline(upper, color='red', linestyle=':', label=f'Üst sınır: {upper:.0f}')
ax.axvline(df_clean['word_count'].mean(), color='orange', linestyle='-', label='Ortalama')
ax.set_xlabel('Kelime Sayısı')
ax.set_ylabel('Frekans')
ax.set_title('Kelime Sayısı Dağılımı + IQR Sınırları')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v/1000)}K'))
ax.legend(fontsize=8)

plt.suptitle('Outlier Analizi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/cleaning_outliers.png', bbox_inches='tight')
plt.show()

print(f'Outlier sayısı: {outlier_mask.sum()} / {len(df_clean)}')
print('Not: Corpus küçük olduğu için outlier eserler korunuyor (model için değerli).')

## 7. Kalite Skoru Dağılımı

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(df_clean['quality_score'], bins=10, color='mediumseagreen', edgecolor='white', alpha=0.85)
ax.set_xlabel('Kalite Skoru')
ax.set_ylabel('Eser Sayısı')
ax.set_title('Kalite Skoru Dağılımı')

ax = axes[1]
qs = df_clean.sort_values('quality_score')
bars = ax.barh(qs['work'], qs['quality_score'],
               color=[PHIL_COLORS.get(p, '#888') for p in qs['philosopher']], alpha=0.85)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
ax.set_xlabel('Kalite Skoru')
ax.set_title('Esere Göre Kalite Skoru')
ax.set_xlim(0, 1.1)

plt.suptitle('Metin Kalite Analizi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/cleaning_quality.png', bbox_inches='tight')
plt.show()

print(f'Ort. kalite skoru : {df_clean["quality_score"].mean():.4f}')
print(f'Min kalite skoru  : {df_clean["quality_score"].min():.4f} ({df_clean.loc[df_clean["quality_score"].idxmin(),"work"]})')
print(f'Max kalite skoru  : {df_clean["quality_score"].max():.4f} ({df_clean.loc[df_clean["quality_score"].idxmax(),"work"]})')

## 8. Temiz Corpus Kaydet

In [ ]:
path = save_clean_corpus(df_clean, output_dir='../data/processed')
print(f'Temiz corpus kaydedildi: {path}')

print('\n=== FAZ 3.3 OZET ===')
print(f'  Ham corpus   : {df_raw.shape[0]} eser, {df_raw["word_count"].sum():,} kelime')
print(f'  Temiz corpus : {df_clean.shape[0]} eser, {df_clean["word_count"].sum():,} kelime')
print(f'  Silinen kel. : {df_raw["word_count"].sum() - df_clean["word_count"].sum():,}')
print(f'  Duplikat     : {df_clean["is_exact_dup"].sum()}')
print(f'  Outlier      : {df_clean["word_count_outlier"].sum()} (korundu)')
print(f'  Ort. kalite  : {df_clean["quality_score"].mean():.4f}')
print('  Sonraki adim : Faz 3.4 — Feature Engineering')